In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

## RAG

In [ ]:
from rag_helper import RAGBase

instructions: str = (
    """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()
)

assistant = RAGBase(
    instructions=instructions,
    index=index,
    llm_client=openai_client,
    model="gpt-5.4-mini",
)

In [9]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from **https://ollama.com/download** for your operating system.
   - **macOS**: download and install the `.pkg`
   - **Windows**: download and install the `.msi`
   - **Linux**: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To test that the local server is running, you can use:
```bash
curl http://localhost:11434
```


In [10]:
messages: list[dict] = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Possibly — but it depends on whether the course is still open for enrollment and what the start date is.\n\nIf you want, I can help you figure it out quickly. To check, you usually need:\n- the course name\n- the provider or platform\n- whether it’s self-paced or instructor-led\n- the current enrollment deadline, if any\n\nIf you’re asking how to message someone, you could say:\n\n> Hi, I just discovered this course and I’m very interested. Is it still possible to join?\n\nIf you share the course details, I can help you draft a more specific message.'

In [ ]:
def search(query: str, **kwargs: dict):

    boost_dict: dict[str, float] = {
        "question": kwargs.get("question", 3.0),
        "section": kwargs.get("section", 0.5),
    }
    filter_dict: dict[str, str] = {
        "course": kwargs.get("course", "llm-zoomcamp"),
    }

    return index.search(
        query,
        num_results=kwargs.get("num_results", 5),
        boost_dict=boost_dict,
        filter_dict=filter_dict,
    )

In [12]:
search_tool: dict = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ.",
            }
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}

In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [14]:
len(response.output)

1

In [15]:
call = response.output[0]

In [16]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join enrollment eligibility late join"}', call_id='call_ku4eYmv6RlAoGyO0uhlTibc0', name='search', type='function_call', id='fc_03d016b11bd32dd9006a342f5c8370819eaf9de4ce841f0977', namespace=None, status='completed')

In [17]:
import json

args = json.loads(call.arguments)
print(json.dumps(args, indent=4))

{
    "query": "join course discovered late can I join enrollment eligibility late join"
}


In [18]:
results = search(**args)

In [19]:
results_json = json.dumps(results, indent=4)

In [20]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": results_json,
}

In [21]:
messages.append(call)

In [22]:
messages.append(function_call_output)

Use message history with tool calls and tool call outputs:

In [23]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [24]:
print(response.output_text)

Yes, you can still join.

If you want a certificate, though, you need to submit your project while the course is still accepting submissions.


In [25]:
usage = response.usage
usage.input_tokens, usage.output_tokens, usage.total_tokens

(772, 33, 805)

In [66]:
def calculate_gpt54mini_price(
    input_tokens: int, output_tokens: int
) -> dict[str, float]:
    """Calculate the cost of using GPT-5.4-mini.

    Parameters
    ----------
    input_tokens: int
        The number of input tokens.
    output_tokens: int
        The number of output tokens.

    Returns
    -------
    dict[str, float]
        The cost of using GPT-5.4-mini.
    """
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION: float = 0.15  # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION: float = 0.60  # $0.60 / 1M output tokens

    input_cost: float = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost: float = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost: float = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


## The Agentic Loop

### Pre-requisites 

In [67]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

In [68]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [69]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join FAQ"}', call_id='call_rjMoPA6EsYDe7BqOhGWYx5Kx', name='search', type='function_call', id='fc_0f7462f73d09c880006a343cddf68081a2be95a820190c1e0f', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"new student discovered course can I join course access FAQ"}', call_id='call_8ZeTJCEhi7eRkS8iSxxCbiBC', name='search', type='function_call', id='fc_0f7462f73d09c880006a343cddf69081a2bb5813e30790a17b', namespace=None, status='completed')]

In [70]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [71]:
messages.extend(response.output)

In [72]:
for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment late join FAQ"}
function_call: search {"query":"new student discovered course can I join course access FAQ"}


In [73]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join FAQ"}', call_id='call_rjMoPA6EsYDe7BqOhGWYx5Kx', name='search', type='function_call', id='fc_0f7462f73d09c880006a343cddf68081a2be95a820190c1e0f', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"new student discovered course can I join course access FAQ"}', call_id='ca

In [74]:
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini", input=messages, tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered late can I join enrollment FAQ"}
function_call: search {"query":"course enrollment discovered late can I still join FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. If you’d like, I can also help you with how to start the course or explain the certificate requirements. Are there other areas you want to explore?


### The full Agentic Loop

In [75]:
# Function Call Helper

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

The full Agent-Loop in one function:

In [76]:
def agent_loop(
    instructions: str,
    question: str,
    model: str = "gpt-5.4-mini",
    max_iters: int = 10,
    print_intermediate: bool = False,
) -> str:
    """The full Agent-Loop in one function.

    Parameters
    ----------
    instructions: str
        The instructions for the agent.
    question: str
        The question to answer.
    model: str
        The model to use.
    max_iters: int
        The maximum number of iterations.

    Returns
    -------
    str
        The answer to the question.
    """
    messages: list[dict] = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model, input=messages, tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                if print_intermediate:
                    print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True
            elif item.type == "message":
                last_answer = item.content[0].text
                if print_intermediate:
                    print("ASSISTANT:")
                    print(last_answer)

        it += 1

        # Done: No function calls or max iterations reached
        if has_function_calls == False or it > max_iters:
            break

    return last_answer

In [77]:
# Developer Prompt

instructions: str = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [78]:
result = agent_loop(instructions, question="How do I run Olama locally?")

iteration #1...
iteration #2...
iteration #3...


In [79]:
print(result)

To run Ollama locally, the FAQ says:

1. Install Ollama from: https://ollama.com/download  
   - macOS: download the `.pkg`
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Start a local model:
   ```bash
   ollama run llama3
   ```
   This downloads the model and starts it locally.

3. To check that the local server is running:
   ```bash
   curl http://localhost:11434
   ```

4. If you want to use it from Python:
   ```bash
   pip install ollama
   ```
   Then:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": your_prompt}]
   )

   print(response['message']['content'])
   ```

If you run into a “connection refused” error in the notebook, the FAQ suggests restarting the Ollama server with:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```

If you want, I can also help with other course setup questions.


## ToyAIKit

In [84]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

SearchResult = list | list[dict[str, str]]

In [85]:
def search(query: str) -> SearchResult:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
    )


search_tool: dict[str, str | dict[str, str]] = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ.",
            }
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}

In [86]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [89]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [90]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [93]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

In [97]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [105]:
print(f"Input cost: {result.cost.input_cost}")
print(f"Output cost: {result.cost.output_cost}")
print(f"Total cost: {result.cost.total_cost}")

Input cost: 0.00108975
Output cost: 0.0013095
Total cost: 0.00239925


Continue from prev. message:

In [106]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [107]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


...

-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='How do I run Docker in Docker?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"Docker in Docker run Docker in Docker